# Taller N.º 8 — Optimización de Spark II: sesgo de datos y memoria

**Sesión 27 · Big Data 2026-II · Universidad Externado de Colombia**

**Carácter del taller:** calificable. El formato —ejecución con inteligencia artificial permitida, corte, cuestionario sin inteligencia artificial y evidencia en capturas— es el que usará el Examen II. Las reglas de material de la Fase 2 son las mismas del Taller 7, sin cambios.

**Qué cambia respecto del Taller 7**

1. Se entregan **tres** capturas obligatorias (C1 a C3), no cuatro.
2. La unidad de observación deja de ser la consulta y pasa a ser la **tarea**: por qué, con el mismo plan y los mismos datos, una tarea tarda mucho más que la mediana de sus compañeras.
3. Durante la Ejecución 4 **se detiene la aplicación y se lanza una segunda**. Eso obliga a consultar la primera en el History Server durante la Fase 2.

**Lo que este taller no ejecuta, pero sí pregunta.** El *broadcast join* se trabajó en la Sesión 26 y no se repite aquí. El cuestionario de la Fase 2 incluye un ítem sobre él, respondible con el notebook publicado de esa sesión, que es material permitido.

---
## Antes de empezar (1): las tres capturas obligatorias

Se exigen únicamente estas tres. Una captura adicional no otorga puntos, y la ausencia de una obligatoria deja sin respaldo las preguntas que dependan de ella.

| N.º | Ejecución | Qué debe verse |
|---|---|---|
| **C1** | E2 | Pestaña **Stages**, página de detalle del *stage* del join: tabla **Summary Metrics for Completed Tasks**, con las filas `Duration` y `Shuffle Read Size / Records` en sus columnas Min, Median y Max |
| **C2** | E3 | Pestaña **SQL / DataFrame**, detalle de la consulta: el recuadro del nodo `AQEShuffleRead` **del lado grande** —el único que incluye las líneas `number of skewed partitions` y `number of skewed partition splits`—, con sus métricas visibles |
| **C3** | E4b | Pestaña **Stages**, página de detalle del *stage* del join bajo memoria reducida: tabla **Summary Metrics**, con las filas `Spill (memory)` y `Spill (disk)` visibles, en la **aplicación B** |

**Criterio de suficiencia**

- En las tres capturas debe verse la **barra de direcciones** del navegador. La URL contiene el identificador `application_…`, que acredita que la ejecución es propia. En C3 ese identificador debe ser el de la **segunda** aplicación.
- Cada imagen se relaciona en su subsección C1, C2 o C3 del archivo de Word, en ese orden.
- Cada captura debe presentar la fecha y hora del sistema.

---
## Antes de empezar (2): cómo identificar el *stage* correcto

La consulta de este taller produce **cuatro *stages*** por ejecución. Dos alimentan el *shuffle* —uno lee el archivo de viajes, otro construye la tabla de zonas— y los otros dos lo consumen.

Hay una trampa: **dos *stages* tienen un número de tareas parecido**. Uno es el del join; el otro es el de la agregación final. No se distinguen por el número de tareas, sino por el volumen:

| | `Shuffle Read` |
|---|---|
| *Stage* del join | cientos de MiB |
| *Stage* de la agregación final | unos pocos KiB |

**El *stage* del join es el de mayor `Shuffle Read` de los cuatro.** Ese, y no otro, es el que se registra y se captura.

Conviene anticipar algo que ocurrirá en la Ejecución 3: si AQE divide alguna partición, el *stage* del join tendrá **más** tareas que las que fija `spark.sql.shuffle.partitions`, mientras que el de la agregación final conservará ese número exacto. Esa diferencia es, en sí misma, una señal.

Como en los talleres anteriores, cada celda incluye una línea `sc.setJobGroup("E#", "...")`, cuya etiqueta aparece en la columna **Description** de las pestañas *Jobs*, *Stages* y *SQL / DataFrame*.

---
## Antes de empezar (3): habrá dos aplicaciones, y la primera se apaga

La memoria de los *executors* no se cambia en caliente: se fija al lanzar la aplicación. Por eso la Ejecución 4 exige detener la aplicación A y lanzar una aplicación B con otra configuración.

- **La aplicación A deja de tener portal vivo.** Sus métricas solo siguen disponibles si el registro de eventos estaba habilitado *en el momento de lanzarla*. El History Server no reconstruye lo que no se escribió: por eso la Ejecución 1 lo verifica **antes** de cualquier otra cosa.
- Durante la Fase 2, la aplicación A se consulta en el **History Server** (`localhost:18080`) y la aplicación B en el **portal vivo** a través de YARN.
- **La aplicación B no se detiene** al terminar la Fase 1. No debe ejecutarse `spark.stop()` ni usarse *Shut Down Kernel*. Cerrar la pestaña del notebook no detiene la aplicación.

---
## Parámetros del taller

Todos los valores que gobiernan las ejecuciones están reunidos en la celda siguiente. No se modifican durante el taller.

In [1]:
# Parametros del taller. No modificar durante la sesion.

RUTA_DATOS = "hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-*.parquet"

# Recursos de la aplicacion
MEMORIA_EXECUTOR_A  = "4g"      # aplicacion A: ejecuciones 1 a 4a
MEMORIA_EXECUTOR_B  = "1g"      # aplicacion B: ejecucion 4b
CORES_EXECUTOR      = "2"
INSTANCIAS_EXECUTOR = "2"

# Ejecuciones 2 y 3: observacion del sesgo
PARTICIONES_SHUFFLE = 200       # es el valor por defecto de Spark
UMBRAL_SESGO        = "8MB"     # skewJoin.skewedPartitionThresholdInBytes (por defecto: 256MB)
FACTOR_SESGO        = "5"       # skewJoin.skewedPartitionFactor (valor por defecto)
TAMANO_OBJETIVO     = "16MB"    # advisoryPartitionSizeInBytes

# Ejecucion 4: presion de memoria
PARTICIONES_PRESION = 4

print("Parametros cargados.")

Parametros cargados.


---
## Ejecución 1 — Sesión, registro de eventos y datos

Se lanza la aplicación A y se verifica que el registro de eventos esté habilitado. Si la verificación no dice `true`, debe informarse a la docente **antes de continuar**: sin ese registro, la aplicación A quedará inaccesible durante la Fase 2 y las preguntas de las Ejecuciones 2 y 3 quedarán sin respaldo.

La tabla de zonas se construye con `spark.range` en lugar de derivarla del archivo de viajes: es idéntica para todos, no gasta una lectura y deja el plan limpio. Los 265 identificadores corresponden a las zonas del servicio de taxis de Nueva York.

**Advertencia sobre el resultado.** La asignación de *borough* es sintética —se reparte por rangos de identificador, igual que en la Sesión 26— y **no corresponde a la geografía real de Nueva York**. Las cifras que imprimirá la consulta son correctas como cálculo distribuido y no admiten ninguna lectura urbana: el propósito de la consulta es mover datos por el *shuffle*, no describir la ciudad.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time


def construir_sesion(nombre, memoria_executor):
    """Lanza una aplicacion sobre YARN con la memoria de executor indicada."""
    return (
        SparkSession.builder
        .appName(nombre)
        .master("yarn")
        .config("spark.executor.memory", memoria_executor)
        .config("spark.executor.cores", CORES_EXECUTOR)
        .config("spark.executor.instances", INSTANCIAS_EXECUTOR)
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
    )


spark = construir_sesion("Taller8 - Sesgo y memoria (aplicacion A)", MEMORIA_EXECUTOR_A)
sc = spark.sparkContext

print("Application ID:", sc.applicationId)
print("Version de Spark:", spark.version)
print("Memoria por executor:", sc.getConf().get("spark.executor.memory"))
print("Cores por executor:", sc.getConf().get("spark.executor.cores"))
print()
print("VERIFICACION DEL REGISTRO DE EVENTOS")
print("  spark.eventLog.enabled:", sc.getConf().get("spark.eventLog.enabled", "no definido"))
print("  spark.eventLog.dir:", sc.getConf().get("spark.eventLog.dir", "no definido"))
print("  Si el valor no es 'true', informar a la docente antes de continuar.")

Application ID: application_1790974913995_0001
Version de Spark: 3.5.7
Memoria por executor: 4g
Cores por executor: 2

VERIFICACION DEL REGISTRO DE EVENTOS
  spark.eventLog.enabled: true
  spark.eventLog.dir: hdfs:///spark-logs
  Si el valor no es 'true', informar a la docente antes de continuar.


In [3]:
sc.setJobGroup("E1", "E1 - Lectura de viajes y construccion de la tabla de zonas")


def preparar_datos(sesion):
    """Devuelve el lado grande (viajes) y el lado pequeno (zonas) del join."""
    df_taxi = sesion.read.parquet(RUTA_DATOS)

    df_viajes = df_taxi.select(
        "VendorID",
        "tpep_pickup_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
    ).filter((F.col("trip_distance") > 0) & (F.col("fare_amount") > 0))

    df_zonas = (
        sesion.range(1, 266)
        .withColumn("LocationID", F.col("id").cast("int"))
        .drop("id")
        .withColumn("zone_name", F.concat(F.lit("Zona_"), F.col("LocationID")))
        .withColumn(
            "borough",
            F.when(F.col("LocationID") <= 100, "Manhattan")
            .when(F.col("LocationID") <= 150, "Brooklyn")
            .when(F.col("LocationID") <= 200, "Queens")
            .otherwise("Other"),
        )
    )
    return df_viajes, df_zonas


df_viajes, df_zonas = preparar_datos(spark)

print("Viajes (lado grande):", df_viajes.count(), "filas")
print("Zonas (lado pequeno):", df_zonas.count(), "filas")

Viajes (lado grande): 39706652 filas
Zonas (lado pequeno): 265 filas


**Registrar en la hoja (fila 1):** Application ID de la aplicación A · número de executors vivos, sin contar el `driver` · cores por executor · valor de `spark.eventLog.enabled`.

*Fuente:* salida de las dos celdas anteriores y pestaña **Executors** del portal.

*No requiere captura.*

---
## La consulta del taller y el calentamiento

Todas las ejecuciones que siguen usan **la misma consulta**: el recaudo y la distancia media por *borough*, que exige unir los 39,7 millones de viajes con la tabla de zonas. Lo único que cambia entre una ejecución y otra es **una condición del entorno**.

**Por qué se ejecuta una vez antes de medir.** La primera corrida de cualquier consulta paga costos que no se repiten: el sistema operativo aún no tiene los archivos en caché, la máquina virtual de Java todavía no ha compilado el código generado y los *executors* acaban de arrancar. Si se compara la primera corrida contra la segunda, parte de la diferencia es ese arranque y no la condición que se quería medir.

Por eso la celda siguiente ejecuta la consulta una vez y **descarta el resultado**. No se registra ni se captura nada de ella.

In [4]:
def consulta(viajes, zonas):
    """Recaudo y distancia media por borough. Identica en todas las ejecuciones."""
    return (
        viajes.join(zonas, viajes["PULocationID"] == zonas["LocationID"], "inner")
        .groupBy("borough")
        .agg(
            F.count("*").alias("viajes"),
            F.round(F.sum("fare_amount"), 2).alias("recaudo_usd"),
            F.round(F.avg("trip_distance"), 3).alias("distancia_media_millas"),
        )
    )


def condiciones(sesion, particiones, corregir_sesgo):
    """Fija el entorno. Entre E2 y E3 solo cambia 'corregir_sesgo'."""
    sesion.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    sesion.conf.set("spark.sql.shuffle.partitions", str(particiones))
    sesion.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
    sesion.conf.set("spark.sql.adaptive.skewJoin.enabled", str(corregir_sesgo).lower())
    sesion.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", FACTOR_SESGO)
    sesion.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", UMBRAL_SESGO)
    sesion.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", TAMANO_OBJETIVO)


def ejecutar(sesion, viajes, zonas):
    """Ejecuta la consulta y devuelve el resultado y el tiempo transcurrido."""
    resultado = consulta(viajes, zonas)
    inicio = time.time()
    filas = resultado.collect()
    return filas, time.time() - inicio


sc.setJobGroup("E1b", "E1b - Calentamiento (se descarta)")
condiciones(spark, PARTICIONES_SHUFFLE, corregir_sesgo=False)
ejecutar(spark, df_viajes, df_zonas)
print("Calentamiento terminado. Su tiempo no se registra.")

[Stage 15:=============================================>        (167 + 4) / 200]

Calentamiento terminado. Su tiempo no se registra.


---
## Ejecución 2 — El sesgo, sin que nadie lo corrija

**Qué es el sesgo.** Al unir dos conjuntos por una clave, Spark reparte las filas entre particiones según el valor de esa clave: **todas las filas con el mismo `PULocationID` caen forzosamente en la misma partición**. En Nueva York unas pocas zonas —los aeropuertos y el centro de Manhattan— concentran una fracción enorme de los viajes, así que esas particiones quedan mucho más cargadas que el resto. El reparto es desigual porque **los datos son desiguales**, no porque el plan esté mal.

El efecto no se ve en el plan. Se ve en las tareas: casi todas terminan en milisegundos y una se queda sola trabajando mientras el *stage* no puede cerrar.

**Por qué esta configuración.** Se apagan tres cosas para que quede una sola variable en juego:

- `autoBroadcastJoinThreshold = -1` fuerza un *sort merge join*. Si la tabla de zonas se difundiera no habría reparto por clave, y por lo tanto no habría sesgo que observar.
- `adaptive.coalescePartitions.enabled = false` evita que AQE fusione particiones pequeñas, lo que movería la mediana contra la cual se compara el máximo.
- `adaptive.skewJoin.enabled = false` desactiva la corrección automática. En esta ejecución interesa el problema, no el remedio.

In [5]:
sc.setJobGroup("E2", "E2 - Join con sesgo sin corregir")
condiciones(spark, PARTICIONES_SHUFFLE, corregir_sesgo=False)

filas_e2, tiempo_e2 = ejecutar(spark, df_viajes, df_zonas)

print("Correccion de sesgo:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
print("Particiones de shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Tiempo total: {:.2f} s".format(tiempo_e2))
print()
for f in sorted(filas_e2, key=lambda r: r["viajes"], reverse=True):
    print("  {:<12} {:>12} viajes   recaudo {:>15,.2f} USD   distancia media {:.3f} mi".format(
        f["borough"], format(f["viajes"], ","), f["recaudo_usd"], f["distancia_media_millas"]))

[Stage 24:===================================================>  (190 + 4) / 200]

Correccion de sesgo: false
Particiones de shuffle: 200
Tiempo total: 8.62 s

  Other          13,595,282 viajes   recaudo  218,024,244.82 USD   distancia media 3.898 mi
  Brooklyn       10,131,916 viajes   recaudo  285,184,177.96 USD   distancia media 6.953 mi
  Queens          8,849,485 viajes   recaudo  150,138,965.71 USD   distancia media 3.654 mi
  Manhattan       7,129,969 viajes   recaudo  133,636,898.62 USD   distancia media 6.093 mi


**Dónde mirar.** En la pestaña *Stages*, esta ejecución aparece en cuatro filas con la etiqueta `E2`. Se abre la página de detalle del *stage* del join, identificado con el criterio de la sección «Antes de empezar (2)»: el de mayor `Shuffle Read`.

Allí, la tabla **Summary Metrics for Completed Tasks** muestra cada métrica en Min, 25th percentile, Median, 75th percentile y Max. Las dos filas que importan son `Duration` y `Shuffle Read Size / Records`.

**Cómo se calcula la razón.** Cada celda de la fila `Shuffle Read Size / Records` contiene **dos cifras separadas por una barra**: a la izquierda el tamaño leído y a la derecha el número de registros. La razón se calcula con el **tamaño**, dividiendo el valor de Max entre el de Median **después de llevar ambos a la misma unidad**. Si el máximo está en MiB y la mediana en KiB, multiplicar los MiB por 1024 antes de dividir; omitir esa conversión cambia el resultado en tres órdenes de magnitud.

La misma razón puede calcularse con el número de registros, y da prácticamente lo mismo. Que ambas coincidan no es casualidad: significa que todas las particiones pesan lo mismo por registro, de modo que el desequilibrio está en cuántas filas recibió cada una y no en lo que ocupa cada fila.

En un reparto equilibrado esa razón se acerca a 1. Conviene calcularla a mano y anotarla: es el valor que sostiene varias preguntas de la Fase 2.

**Un segundo hallazgo, en la pestaña *SQL / DataFrame*.** El plan contiene **tres nodos `Exchange`**: uno por cada lado del join y un tercero, más abajo, donde se reúnen los resultados parciales de la agregación. El que interesa es el del **lado grande**, situado justo debajo del `Scan parquet`, y se reconoce sin ambigüedad por una cifra de su propio recuadro: su `shuffle records written` coincide con el número total de viajes, unos 39,7 millones. El del lado pequeño escribe 265 registros y el de la agregación, unos pocos cientos.

En ese recuadro hay dos cifras que no coinciden: `data size total`, que es lo que pesan las filas en memoria, y `shuffle bytes written total`, que es lo que realmente se escribió en disco. La segunda es varias veces menor que la primera, porque el *shuffle* va comprimido y el reparto por clave agrupa valores iguales, que se comprimen bien.

**Registrar en la hoja (fila 2):** número de tareas del *stage* · `Duration` en Max y Median · `Shuffle Read Size / Records` en Max y Median · razón máximo ÷ mediana del **tamaño** leído · `data size total` y `shuffle bytes written total` del `Exchange` del lado grande · tiempo total impreso por la celda.

*Fuente:* pestaña **Stages**, página de detalle del *stage*; pestaña **SQL / DataFrame**, recuadro del `Exchange` cuyo `shuffle records written` ronda los 39,7 millones; y salida de la celda anterior.

**Captura C1:** la tabla *Summary Metrics* de la página de detalle del *stage*, con la barra de direcciones a la vista.

---
## Ejecución 3 — El mismo join, con la corrección activa

Se repite la ejecución anterior cambiando **una sola condición**: `adaptive.skewJoin.enabled` pasa a `true`.

**Cómo decide AQE que una partición está sesgada.** Deben cumplirse dos condiciones a la vez:

1. Que la partición supere la mediana de las demás multiplicada por `skewedPartitionFactor` (valor por defecto: 5).
2. Que además supere `skewedPartitionThresholdInBytes`, un umbral absoluto cuyo valor por defecto es **256 MB**.

El segundo requisito es el que obliga a ajustar la configuración en este clúster. Un umbral de 256 MB está pensado para volúmenes de producción; a la escala de esta sala ninguna partición lo alcanza, de modo que el mecanismo no se activaría nunca y quedaría invisible. Bajarlo no cambia la lógica: la hace observable. Aquí se usa el valor de `UMBRAL_SESGO` declarado en los parámetros.

**Qué hace cuando la detecta.** No redistribuye los datos: parte la partición grande en subparticiones del tamaño fijado en `advisoryPartitionSizeInBytes` y replica, en cada una, las filas correspondientes del otro lado del join. Donde había una tarea muy larga quedan varias tareas cortas en paralelo.

Por eso el mecanismo **solo aplica a joins con shuffle** —*sort merge join* y *shuffle hash join*—. Un *broadcast join* nunca se optimiza por sesgo, y la razón es la de la Sesión 26: no reparte por clave, así que no tiene particiones sesgadas que partir.

In [6]:
sc.setJobGroup("E3", "E3 - Join con correccion de sesgo activa")
condiciones(spark, PARTICIONES_SHUFFLE, corregir_sesgo=True)

filas_e3, tiempo_e3 = ejecutar(spark, df_viajes, df_zonas)

print("Correccion de sesgo:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))
print("Umbral absoluto:", spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes"))
print("Factor:", spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionFactor"))
print("Tamano objetivo de particion:", spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes"))
print("Tiempo total: {:.2f} s".format(tiempo_e3))
print()
print("Comparacion con E2: {:.2f} s  ->  {:.2f} s".format(tiempo_e2, tiempo_e3))

[Stage 29:==============================================>       (174 + 4) / 204]

Correccion de sesgo: true
Umbral absoluto: 8MB
Factor: 5
Tamano objetivo de particion: 16MB
Tiempo total: 8.51 s

Comparacion con E2: 8.62 s  ->  8.51 s


**Dónde mirar.** Esta vez la evidencia está en la pestaña **SQL / DataFrame**. Al abrir el detalle de la consulta etiquetada `E3` aparecen **dos recuadros `AQEShuffleRead`**, uno por cada lado del join. Los dos se titulan igual y los dos reportan el mismo `number of partitions`, así que el título no sirve para distinguirlos. Se distinguen por su contenido:

| | Lado grande (el que interesa) | Lado pequeño |
|---|---|---|
| `partition data size total` | cientos de MiB | unos pocos KiB |
| `number of skewed partitions` | aparece | no aparece |
| `number of skewed partition splits` | aparece | no aparece |

**Las dos últimas líneas son el criterio: solo el recuadro del lado grande las tiene.**

En la sección **Details**, debajo del DAG, hay una segunda confirmación en el texto del plan. Allí sí aparece la palabra: el nodo se lee `AQEShuffleRead skewed`, y la línea del join dice `SortMergeJoin(skew=true)`. En la Ejecución 2 esa misma línea dice solo `SortMergeJoin`.

Conviene además volver a *Stages* y comparar el número de tareas del *stage* del join con el de la Ejecución 2. Los tres números —particiones sesgadas, divisiones aplicadas y tareas— guardan entre sí una relación aritmética exacta que conviene encontrar.

**Registrar en la hoja (fila 3):** `number of partitions` · `number of skewed partitions` · `number of skewed partition splits` · `partition data size total` en sus tres cifras · número de tareas del *stage* del join, frente al de la fila 2 · `Duration` en Max y Median · tiempo total impreso por la celda.

*Fuente:* pestaña **SQL / DataFrame**, recuadro del nodo `AQEShuffleRead` del lado grande y sección *Details*; pestaña **Stages**.

**Captura C2:** ese recuadro del lado grande, con sus métricas visibles y con la barra de direcciones a la vista.

*Comparación obligatoria:* la razón máximo ÷ mediana de la fila `Duration`, calculada con los valores de la fila 2 de la hoja y de nuevo con los de la fila 3. Si la corrección actuó, el máximo se acercó algo a la mediana, porque la tarea más larga quedó repartida entre varias. Y una segunda observación, menos cómoda: **el tiempo total no mejora de forma apreciable, y puede incluso resultar algo mayor que en la Ejecución 2**. Conviene tener preparada una explicación de por qué la estructura del *stage* cambió y el reloj no lo refleja.

---
## Ejecución 4 — La memoria: lo que no se cambia en caliente

Las tres ejecuciones anteriores modificaron el comportamiento de Spark con `spark.conf.set`, sin reiniciar nada. Todas ellas son configuraciones **de consulta**: gobiernan cómo se planifica y se reparte el trabajo.

La memoria del *executor* es de otra clase. Es una configuración **de infraestructura**: define cuánta memoria pide YARN al crear cada contenedor. Cuando la aplicación ya está corriendo, los contenedores existen y su tamaño está fijado.

La celda siguiente intenta cambiarla en caliente y muestra qué ocurre. Después ejecuta un **trabajo de presión**: la misma consulta, repartida en muy pocas particiones. Con `PARTICIONES_PRESION` particiones, cada tarea debe ordenar en memoria una porción enorme del total. Cuando esa porción no cabe, Spark no falla: **derrama** a disco (*spill*) y termina igual.

Conviene no dar por hecho qué se observará. El derrame se paga en entrada y salida, pero ese costo puede quedar oculto detrás de otras etapas más lentas, de modo que **el reloj no es un buen indicador**. Lo que se busca en esta ejecución son las métricas de memoria y de derrame, no la duración.

**Una advertencia sobre la comparación que viene.** Sería cómodo suponer que con memoria holgada no hay derrame y con memoria escasa sí. La realidad suele ser menos limpia: con esta presión es probable que **ambas ejecuciones derramen**, y entonces la pregunta deja de ser *si* derramaron y pasa a ser **cuánto y cuántas tareas**. La columna `Min` de la tabla es la que responde a lo segundo: si vale cero, alguna tarea se salvó; si no, ninguna.

Nótese que con tan pocas particiones el sesgo **desaparece de la vista**: 263 claves repartidas en cuatro grupos se promedian entre sí. Es el mismo dato y el mismo plan; lo que cambió es la resolución con que se observa.

In [7]:
print("Intento de cambio en caliente de la memoria del executor:")
try:
    spark.conf.set("spark.executor.memory", MEMORIA_EXECUTOR_B)
    print("  La instruccion no produjo error.")
except Exception as error:
    print("  La instruccion produjo un error:", type(error).__name__)

print("  Memoria efectiva de los executors en curso:",
      sc.getConf().get("spark.executor.memory"))
print("  Conclusion: el valor con el que se lanzo la aplicacion no cambio.")
print()

sc.setJobGroup("E4a", "E4a - Trabajo de presion en la aplicacion A")
condiciones(spark, PARTICIONES_PRESION, corregir_sesgo=False)

filas_e4a, tiempo_e4a = ejecutar(spark, df_viajes, df_zonas)

print("Aplicacion A:", sc.applicationId)
print("Memoria por executor:", sc.getConf().get("spark.executor.memory"))
print("Particiones de shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Tiempo total: {:.2f} s".format(tiempo_e4a))

Intento de cambio en caliente de la memoria del executor:
  La instruccion produjo un error: AnalysisException
  Memoria efectiva de los executors en curso: 4g
  Conclusion: el valor con el que se lanzo la aplicacion no cambio.



[Stage 38:===========================================>              (3 + 1) / 4]

Aplicacion A: application_1790974913995_0001
Memoria por executor: 4g
Particiones de shuffle: 4
Tiempo total: 11.87 s


**Registrar en la hoja (fila 4):** número de tareas del *stage* del join · `Shuffle Read Size / Records` en Median y Max, y su razón · `Peak Execution Memory` en Max · `Spill (memory)` en **Min y Max** · `Spill (disk)` en **Min y Max** · `GC Time` en Max · tiempo total.

*Fuente:* pestaña **Stages**, página de detalle del *stage* del join, tabla *Summary Metrics*.

*No requiere captura.* Esta fila es el término de comparación de la siguiente.

> Si las filas de *spill* no aparecen en la tabla, significa que no hubo derrame en ninguna tarea. Debe registrarse así, con la palabra **ninguno**. Un valor ausente también es un dato, siempre que se indique dónde se buscó.

### Ejecución 4b — La misma presión con menos memoria

Ahora se detiene la aplicación A y se lanza la aplicación B, idéntica en todo salvo en la memoria de cada *executor*. Se repite exactamente el mismo trabajo de presión.

**Antes de ejecutar la celda siguiente, conviene tener ya tomados todos los valores de las filas 1 a 4 de la hoja.** A partir de este punto el portal de la aplicación A deja de responder y sus métricas solo se recuperan por el History Server.

La aplicación B arranca en frío, así que también se ejecuta un calentamiento antes de medir.

**Un indicio adicional, fuera del portal de Spark.** En el portal de YARN, el porcentaje de uso del clúster cambia entre las dos aplicaciones, mientras que los cores por *executor* siguen siendo los mismos. Ese porcentaje se calcula sobre la **memoria reservada** —la del *driver* más la de los *executors*, redondeada por YARN al tamaño de contenedor que concede—, no sobre los núcleos. Es la confirmación, desde fuera de Spark, de que entre las dos aplicaciones cambió una sola variable: el paralelismo es idéntico, y por eso las cuatro tareas siguen ejecutándose a la vez y leen exactamente lo mismo.

In [8]:
spark.stop()
print("Aplicacion A detenida.")
time.sleep(5)

spark = construir_sesion("Taller8 - Sesgo y memoria (aplicacion B)", MEMORIA_EXECUTOR_B)
sc = spark.sparkContext

print("Aplicacion B:", sc.applicationId)
print("Memoria por executor:", sc.getConf().get("spark.executor.memory"))
print("spark.eventLog.enabled:", sc.getConf().get("spark.eventLog.enabled", "no definido"))

df_viajes, df_zonas = preparar_datos(spark)

sc.setJobGroup("E4b0", "E4b0 - Calentamiento de la aplicacion B (se descarta)")
condiciones(spark, PARTICIONES_PRESION, corregir_sesgo=False)
ejecutar(spark, df_viajes, df_zonas)
print("Calentamiento terminado. Su tiempo no se registra.")

Aplicacion A detenida.
Aplicacion B: application_1790974913995_0002
Memoria por executor: 1g
spark.eventLog.enabled: true


[Stage 5:============================================>              (3 + 1) / 4]

Calentamiento terminado. Su tiempo no se registra.


In [9]:
sc.setJobGroup("E4b", "E4b - Trabajo de presion con memoria reducida")
condiciones(spark, PARTICIONES_PRESION, corregir_sesgo=False)

filas_e4b, tiempo_e4b = ejecutar(spark, df_viajes, df_zonas)

print("Aplicacion B:", sc.applicationId)
print("Tiempo total: {:.2f} s".format(tiempo_e4b))
print()
print("Mismo trabajo, distinta memoria:")
print("  {} por executor: {:.2f} s".format(MEMORIA_EXECUTOR_A, tiempo_e4a))
print("  {} por executor: {:.2f} s".format(MEMORIA_EXECUTOR_B, tiempo_e4b))

[Stage 14:===========================================>              (3 + 1) / 4]

Aplicacion B: application_1790974913995_0002
Tiempo total: 12.02 s

Mismo trabajo, distinta memoria:
  4g por executor: 11.87 s
  1g por executor: 12.02 s


**Registrar en la hoja (fila 5):** Application ID de la aplicación B · `Shuffle Read Size / Records` en Median y Max · `Peak Execution Memory` en Max · `Spill (memory)` en **Min y Max** · `Spill (disk)` en **Min y Max** · `GC Time` en Max · tiempo total.

*Fuente:* pestaña **Stages** de la aplicación B, página de detalle del *stage* del join.

**Captura C3:** la tabla *Summary Metrics* de esa página, con las filas `Spill (memory)` y `Spill (disk)` visibles y con la barra de direcciones mostrando el identificador de la **aplicación B**.

*Comparación obligatoria:* entre la fila 4 y la fila 5, el `Shuffle Read` debe ser idéntico —son los mismos datos, repartidos igual—. Lo que cambia está en tres cifras: el `Peak Execution Memory`, la columna `Min` del derrame y el `Max` del derrame. Debe poder explicarse qué significa cada una de esas tres diferencias, y adónde fue a parar lo que ya no cupo en memoria.

*Y una advertencia sobre el tiempo:* la aplicación B se lanzó de cero, así que parte de su duración es arranque. **El tiempo no es la evidencia aquí; el *spill* sí.**

---
## Cierre de la Fase 1

Antes del corte, verificar que:

1. La hoja de registro está completa, con la fuente de cada valor y el número de captura donde corresponde.
2. El archivo de Word contiene las tres capturas, numeradas C1 a C3, y en C3 se ve el identificador de la aplicación B.
3. El archivo se convirtió a PDF y se entregó por Microsoft Teams.

**Durante la Fase 2:**

- **No ejecutar `spark.stop()`** ni apagar el kernel. La aplicación B debe seguir viva.
- **Cerrar la pestaña de este notebook.** El kernel sigue activo en el servidor y el portal continúa respondiendo.
- La aplicación A se consulta en el History Server (`localhost:18080`); la aplicación B, en el portal vivo a través de YARN.
- El cuestionario se responde **en papel**. En pantalla solo puede estar el portal de Spark.
- La captura nueva que exige el ítem abierto se carga en Microsoft Forms al cerrar la fase.

**Material permitido en la Fase 2** (el mismo del Taller 7, sin cambios): portal de Spark de las aplicaciones propias, únicamente para consulta; hoja de registro diligenciada; archivo de capturas entregado en el corte, únicamente para consulta; notebooks de sesiones anteriores en la versión publicada por la docente en Microsoft Teams; apuntes propios manuscritos.

**No permitido:** herramientas de inteligencia artificial; copias propias o editadas de los notebooks; apuntes digitales; el notebook de este taller, que debe permanecer cerrado.

In [ ]:
# ATENCION: esta linea debe permanecer comentada durante todo el taller.
# Ejecutarla cierra la aplicacion B y deja el portal inaccesible,
# lo que impide responder el cuestionario de la Fase 2.
# spark.stop()